<a href="https://colab.research.google.com/github/Pensive1881/DSR44_2025/blob/main/2_most_common_bugs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Please, make a copy of the notebook.
import gdown
import numpy as np
import os
import pandas as pd
import torch

from torch import nn
from torch.utils.tensorboard import SummaryWriter
from sklearn.datasets import make_regression

%matplotlib inline
%load_ext tensorboard

##### Load tensorboard logs and create some helper functions.

In [ ]:
url = f'https://drive.google.com/uc?id=1ww5f0TvZW6OAF6ePf2iV3lfB99pBx2y3'

output = 'tb_logs.zip'
gdown.download(url, output, quiet=False)

# unzip
!unzip tb_logs.zip
!rm tb_logs.zip

In [ ]:
def make_writer(filepath, dir_name):
    """ Creates a directory to save tensorboard events """
    path = os.path.join(filepath, dir_name)
    os.makedirs(path, exist_ok=True)
    print(f'Creating a tensorboard directory: {path}')
    writer =SummaryWriter(log_dir=path)
    return writer

def standardize(array):
    means = np.mean(array, axis=0)
    std = np.std(array, axis=0)
    return (array - means) /  std

# Most common bugs

## Resources

- [Chapter 4 of Deep learning book. Numerical computation](https://www.deeplearningbook.org/contents/numerical.html)
- [Gradient norm clipping](http://proceedings.mlr.press/v28/pascanu13.html)

## Incorrect tensor shapes

### Most common reasons:

- Flipped dimensions when using tf.reshape.
- Sum, avg, softmax over wrong dimension.
- Forgot to flatten after conv layers.
- Forgot to get rid of extra "1" dimensions, e.g. if shape is (None, 1, 1, 4).

In Pytorch, as well as in other libraries like numpy and tensorflow, you can accidentally broadcast tensors and then it can fail silently or just output wrong results.

In [ ]:
y_true = np.array([0.1, 0.7, 0.02, 0.08, 0.05, 0.05])
y_true_extra_dim = np.expand_dims(y_true, -1)
y_pred = np.array([0.1, 0.6, 0.05, 0.05, 0.1, 0.1])

In [ ]:
print(f'y_true: {y_true} \n')
print(f'Shape of y_true: {y_true.shape} \n')
print(f'y_true_extra_dim: {y_true_extra_dim} \n')
print(f'Shape of y_true_extra_dim: {y_true_extra_dim.shape} \n')

In [ ]:
y_pred

In [ ]:
y_pred.shape

Say we want to divide y_true by y_pred. What shapes do we expect to get?

In [ ]:
y_true / y_pred

In [ ]:
y_true_extra_dim / y_pred

#### KL-divergence

KL-divergence is used in some models like VAEs or Bayesian models.

In [ ]:
kl = torch.nn.KLDivLoss(reduction="batchmean", log_target=False)

print(f'KLD for y_true: {kl(torch.log(torch.tensor(y_pred)), torch.tensor(y_true))}')
print(f'KLD for y_true_extra_dim: {kl(torch.log(torch.tensor(y_pred)), torch.tensor(y_true_extra_dim))}')

## Pre-processing inputs incorrectly

- Forgot to standardize/scale.
    -  It makes the resulting model dependent on the choice of units used in the input.
- Too much augmentation.


### Regression example with Auto MPG data

#### Load the data and create a pandas DataFrame

In [ ]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = ['MPG','Cylinders','Displacement','Horsepower','Weight',
                'Acceleration', 'Model Year', 'Origin']
dataset = pd.read_csv(url, names=column_names,
                      na_values = "?", comment='\t',
                      sep=" ", skipinitialspace=True)
dataset = dataset.dropna()
dataset.drop('Origin', axis=1, inplace=True)

#### Create labels and train set

In [ ]:
labels = dataset.pop('MPG')
labels = np.array(labels).astype('float32')
labels = standardize(labels)

Let's make the difference in scales for some features even more pronounced and print the statistics.

In [ ]:
dataset['Horsepower'] = dataset['Horsepower'] * 1000
dataset['Displacement'] = dataset['Displacement'] / 1000
train_set = np.array(dataset).astype('float32')
train_set_stand = standardize(train_set)

In [ ]:
print(f"Train_set std:")
{col: avg for col, avg in zip(dataset.columns, train_set.std(axis=0))}

In [ ]:
print(f"Train_set_stand std:")
{col: avg for col, avg in zip(dataset.columns, train_set_stand.std(axis=0))}

In [ ]:
train_set_not_scaled = torch.utils.data.TensorDataset(torch.tensor(train_set), torch.tensor(labels.reshape(-1, 1)))
train_set_scaled = torch.utils.data.TensorDataset(torch.tensor(train_set_stand), torch.tensor(labels.reshape(-1, 1)))

In [ ]:
train_loader_not_scaled = torch.utils.data.DataLoader(train_set_not_scaled, batch_size=32, shuffle=True)
train_loader_scaled = torch.utils.data.DataLoader(train_set_scaled, batch_size=32, shuffle=True)

### Model

In [ ]:
class RegressorNet(nn.Module):
    """
    A class for solving regression problems.
    """

    def __init__(self, in_dim):
        super().__init__()

        self.regressor = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.regressor(x)

In [ ]:
net_not_scaled = RegressorNet(in_dim=train_set.shape[1])
net_scaled = RegressorNet(in_dim=train_set.shape[1])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net_not_scaled.to(device)
net_scaled.to(device)
print(net_not_scaled)

In [ ]:
optimizer_not_scaled = torch.optim.Adam(net_not_scaled.parameters())
optimizer_scaled = torch.optim.Adam(net_scaled.parameters())

mse_loss = nn.MSELoss()

### Train and output to Tensorboard

In [ ]:
def train(model, optimizer, loss_fn, loader, epochs, save_dir):

    writer = make_writer(os.path.join('tb_logs'), save_dir)

    for epoch in range(0, epochs + 1):
        model.train()
        train_loss = 0

        if epoch % 100 == 0:
            print('Epoch {} is running...'.format(epoch))

        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(loader)

        if epoch % 100 == 0:
            print(f'{train_loss}')

        # Log to TB
        # Loss
        writer.add_scalar("Loss/Train", train_loss, epoch)

        # Grads
        for name, param in model.named_parameters():
            writer.add_histogram(f"Gradients/{name}", param.grad, epoch)

        # Grad norms
        for name, param in model.named_parameters():
            grad_norm = param.grad.norm().item()
            writer.add_scalar(f"Gradient_Norms/{name}", grad_norm, epoch)

        # Flush the writer to ensure all data is written
        writer.flush()



In [ ]:
#train(net_not_scaled, optimizer_not_scaled, mse_loss, train_loader_not_scaled, 1000, 'scaling/regression_non_standard')
#train(net_scaled, optimizer_scaled, mse_loss, train_loader_scaled, 1000, 'scaling/regression_standard')

In [ ]:
%tensorboard --logdir tb_logs/scaling --port 6006

## Incorrect input to the loss/ incorrect loss

- Softmaxed outputs to a loss that expects logits or vice-versa.
- E.g. MSE loss when categorical loss is expected.
- ReLU in the last layer for regression problems.

### Wrong Loss Function

In [ ]:
ce_loss = nn.CrossEntropyLoss()

net_correct_loss = RegressorNet(in_dim=train_set.shape[1])
net_wrong_loss = RegressorNet(in_dim=train_set.shape[1])

net_correct_loss.to(device)
net_wrong_loss.to(device)

optimizer_correct_loss = torch.optim.Adam(net_correct_loss.parameters())
optimizer_wrong_loss = torch.optim.Adam(net_wrong_loss.parameters())

In [ ]:
#train(net_correct_loss, optimizer_correct_loss, mse_loss, train_loader_scaled, 1000, "loss_mismatch/correct_loss")
#train(net_wrong_loss, optimizer_wrong_loss, ce_loss, train_loader_scaled, 1000, "loss_mismatch/wrong_loss")

In [ ]:
%tensorboard --logdir tb_logs/loss_mismatch --port 6007

### Zero loss explanation.

Why is the loss zero at epoch 0 already? Let's dive deeper into the [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) calculation.

If you open the torch docs you'll see that the Cross Entropy loss has 2 operation modes, based on the `dtype` and `shape` of your target tensor. In our case it interprets the inputs and targets as softmaxed probabilities with tensor entries for each data point corresponding to a class label. Let's see their shapes.

In [ ]:
batch = next(iter(train_loader_scaled))
x, y_true = batch

In [ ]:
y_pred = net_wrong_loss(x)

print(f"Shape of y_pred: {y_pred.shape}, Dtype: {y_pred.dtype}")
print(f"Shape of y_true: {y_true.shape}, Dtype: {y_true.dtype}")

The both shapes are [batch_size, 1], so the loss "assumes" that there's only **1** class. In that case our softmax becomes:

$$\text{softmax}(x) = \frac{exp(x_0)}{exp(x_0)} = 1$$

and the LogSoftmax operation produces a zero output for each data point in the batch:

$$\log(1) = 0$$.

this, in turn, zeros out the whole loss:

$$L(x, y) = -\sum_j^{C-1} y_{true, j} \cdot \log(\text{softmax}(x_j)) = y_0 \cdot \log(1) = 0$$

### ReLU in the last layer

In [ ]:
# Create models
class RegressorNetWithRelu(nn.Module):
    """
    A class for solving regression problems.
    """

    def __init__(self, in_dim):
        super().__init__()

        self.regressor = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.regressor(x)

In [ ]:
net =  RegressorNet(in_dim=train_set.shape[1])
net_with_relu = RegressorNetWithRelu(in_dim=train_set.shape[1])
print(f"Correct architecture:\n {net}")
print(f"Incorrect architecture:\n {net_with_relu}")

net.to(device)
net_with_relu.to(device)

optimizer = torch.optim.Adam(net.parameters())
optimizer_with_relu = torch.optim.Adam(net_with_relu.parameters())

In [ ]:
def train_extra_logs(model, optimizer, loss_fn, loader, epochs, save_dir):

    writer = make_writer(os.path.join('tb_logs'), save_dir)

    for epoch in range(0, epochs + 1):
        model.train()
        train_loss = 0

        if epoch % 100 == 0:
            print('Epoch {} is running...'.format(epoch))

        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(loader)

        if epoch % 100 == 0:
            print(f'{train_loss}')

        # Log to TB
        # Loss
        writer.add_scalar("Loss/Train", train_loss, epoch)

        # Grads
        for name, param in model.named_parameters():
            writer.add_histogram(f"Gradients/{name}", param.grad, epoch)

        # Grad norms
        for name, param in model.named_parameters():
            grad_norm = param.grad.norm().item()
            writer.add_scalar(f"Gradient_Norms/{name}", grad_norm, epoch)

        # Weight hists
        for name, param in model.named_parameters():
            writer.add_histogram(f"Weights/{name}", param.data, epoch)

        # Weight norms
        for name, param in model.named_parameters():
            weight_norm = param.data.norm().item()
            writer.add_scalar(f"Weight_Norms/{name}", weight_norm, epoch)

        # Output hists
        writer.add_histogram("Outputs/Train", y_pred, epoch)


        # Flush the writer to ensure all data is written
        writer.flush()

In [ ]:
#train_extra_logs(net, optimizer, mse_loss, train_loader_scaled, 1000, 'last_layer_relu/no_relu')
#train_extra_logs(net_with_relu, optimizer_with_relu, mse_loss, train_loader_scaled, 1000, 'last_layer_relu/with_relu')

In [ ]:
%tensorboard --logdir tb_logs/last_layer_relu --port 6008

## Numerical instabilities

- Vanishing and exploding gradients.
- Softmax over a very large value.
- Operations including divisions by values close to zero.
- Big policy updates in RL.

#### Exploding gradients and gradient clipping

In [ ]:
M = torch.randn((4, 4))
print(f'A single matrix \n \n {M.numpy()}')
for i in range(100):
    M = M @ torch.randn((4, 4))

print(f'\nAfter multiplying 100 matrices \n \n {M.numpy()}')

#### Gradient clipping

- Clip a gradient by norm:
$\textbf{g} \gets \frac{\theta}{||\textbf{g}||}\textbf{g} $
    - For example: $$\textbf{g}= [-2, 3, 6]$$ $$\theta = 5$$ $$||\textbf{g}|| = 7$$ $$\textbf{g} \gets [-2, 3, 6]\cdot \frac{5}{7}$$
    
- Clip gradient by value:
    - If $g_i < \theta_1$, then $g_i \gets \theta_1$ and $g_i > \theta_2$, then $g_i \gets \theta_2$
    - For example: $$\textbf{g}= [-2, 3, 10]$$ $$\theta_1 = 0, \theta_2 = 5$$  $$ \textbf{g} \gets [0, 3, 5]$$

    
- Clip gradient by global norm:
    - Rescales a list of tensors so that the total norm of the vector of all their norms does not exceed a threshold.
    - For example: $$\textbf{g}_1 = [-2, 3, 6]$$ $$\textbf{g}_2= [-4, 6, 12]$$ $$\theta = 14$$ $$||\textbf{g}_1|| = 7$$ $$||\textbf{g}_2|| = 14$$ $$\textbf{g}_1 \gets [-2, 3, 6]\cdot \frac{14}{\sqrt{7^2 + 14^2}}$$ $$\textbf{g}_2 \gets [-4, 6, 12]\cdot \frac{14}{\sqrt{7^2 + 14^2}} $$
    

#### General regression dataset example

In [ ]:
# generate regression dataset
x, y = make_regression(n_samples=1000, n_features=20, noise=0.1, random_state=1)
# split into train and test
n_train = 800
trainX = x[:n_train, :].astype('float32')
trainY = y[:n_train].astype('float32').reshape(-1, 1)
testX = x[n_train:, :].astype('float32')
testY = y[n_train:].astype('float32').reshape(-1, 1)

# Creat Datasets & Loaders
train_dataset = torch.utils.data.TensorDataset(torch.tensor(trainX), torch.tensor(trainY))
test_dataset = torch.utils.data.TensorDataset(torch.tensor(testX), torch.tensor(testY))

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Plot data
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(6, 6))
plt.hist(trainY[:, 0], bins=30)
plt.show(fig)

In [ ]:
net_not_clipped = RegressorNet(in_dim=trainX.shape[1])
net_clipped = RegressorNet(in_dim=trainX.shape[1])


net_not_clipped.to(device)
net_clipped.to(device)

optimizer_clipped = torch.optim.SGD(net_clipped.parameters(), lr=5e-3, momentum=0.9)
optimizer_not_clipped = torch.optim.SGD(net_not_clipped.parameters(), lr=5e-3, momentum=0.9)

In [ ]:
def train_with_clipping(model, optimizer, loss_fn, loader, epochs, save_dir, clip=True):

    writer = make_writer(os.path.join('tb_logs'), save_dir)

    for epoch in range(0, epochs + 1):
        model.train()
        train_loss = 0

        if epoch % 10 == 0:
            print('Epoch {} is running...'.format(epoch))

        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            loss.backward()
            # Clip gradients
            if clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(loader)

        if epoch % 10 == 0:
            print(f'{train_loss}')

        # Log to TB
        # Loss
        writer.add_scalar("Loss/Train", train_loss, epoch)

        # Grads
        for name, param in model.named_parameters():
            if not torch.any(torch.isnan(param.grad)):
                writer.add_histogram(f"Gradients/{name}", param.grad, epoch)

        # Grad norms
        for name, param in model.named_parameters():
            if not torch.any(torch.isnan(param.grad)):
                grad_norm = param.grad.norm().item()
                writer.add_scalar(f"Gradient_Norms/{name}", grad_norm, epoch)

        # Flush the writer to ensure all data is written
        writer.flush()



In [ ]:
#train_with_clipping(net_not_clipped, optimizer_not_clipped, mse_loss, train_loader, 100, 'exploding_grads/no_clipping', clip=False)
#train_with_clipping(net_clipped, optimizer_clipped, mse_loss, train_loader, 100, 'exploding_grads/clipping', clip=True)

In [ ]:
%tensorboard --logdir tb_logs/exploding_grads --port 6009

## Wrong model modes: model.train() vs model.eval()

Often during running a model on the validation/test sets people forget to set the model into the evaluation mode that turns off the components of the model that are made to behave randomly at training time:

- Dropout
- Batch normalization
- Custom logic for a `nn.Module` that behaves differently in `train` and `eval` modes.

### Dropout

Dropout randomly zeros out a portion of neuron outputs during training, essentially turning some of its neurons 'off' for that specific forward/backward pass. When we forget to turn it off during evaluation, we degrade the performance.

### BatchNorm

In `train` mode, it does the following:
 - Calculates batch statistics: mean and variance of the current mini-batch.
 - Uses this mini-batch mean and variance to normalize the activations.
 - Updates its internal `running mean` and `running var` to keep a long-term average of the statistics it sees during training.

In `eval` mode:

- Freezes the statistics
- Uses its saved `running mean` and `running var` to normalize the activations.

$$\hat{x} = \frac{x - \mathbb{E}[x]}{\sqrt{Var(x) + \epsilon}}$$

### Custom train & eval logic

Any custom logic that can work differently in those two modes and would lead to an error if we forget to switch between them.



Let's see how our model works on the test set if we toggle the `eval` mode on/off.

In [ ]:
with torch.no_grad():
    net_clipped.eval()
    test_loss = 0

    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        y_pred = net_clipped(x)
        loss = mse_loss(y_pred, y)
        test_loss += loss.item()

    test_loss /= len(test_loader)
    print(f'Test loss with eval mode on: {test_loss}')

    net_clipped.train()
    test_loss = 0

    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        y_pred = net_clipped(x)
        loss = mse_loss(y_pred, y)
        test_loss += loss.item()

    test_loss /= len(test_loader)
    print(f'Test loss with eval mode off: {test_loss}')


## OOM errors

### Common issues and causes

- Too big a tensor:
    - Too large a batch size for your model
    - Too many fully connected layers
- Too much data:
    - Loading a dataset that is too big into memory
    - Allocating too large a buffer for dataset creation
- Duplicating operations:
    - Memory leak due to creating multiple models at the same time
    - Repeatedly creating an operation (e.g. in a function that gets called many times)
- Other processes:
    - Other processes taking GPU memory